In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Imports section
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import PolynomialFeatures

## Part 1. Loading the dataset

In [3]:
# Using pandas load the dataset (load remotely, not locally)
df = pd.read_csv("/content/drive/MyDrive/University/Academics/Fall 2024/CS 383/Projects/CS383_P4/science_data_large.csv")
# Output the first 15 rows of the data
display(df.head(15))
# Display a summary of the table information (number of datapoints, etc.)
df.shape
df.info()
df.describe()

,Temperature °C,Mols KCL,Size nm^3
0,469,647,6.244743e+05
1,403,694,5.779610e+05
2,302,975,6.196847e+05
3,779,916,1.460449e+06
4,901,18,4.325726e+04
5,545,637,7.124634e+05
6,660,519,7.006960e+05
7,143,869,2.718260e+05
8,89,461,8.919803e+04
9,294,776,4.770210e+05


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Temperature °C  1000 non-null   int64  
 1   Mols KCL        1000 non-null   int64  
 2   Size nm^3       1000 non-null   float64
dtypes: float64(1), int64(2)
memory usage: 23.6 KB


,Temperature °C,Mols KCL,Size nm^3
count,1000.000000,1000.000000,1.000000e+03
mean,500.500000,471.530000,5.086111e+05
std,288.819436,288.482872,4.474838e+05
min,1.000000,1.000000,1.611429e+01
25%,250.750000,226.750000,1.298267e+05
50%,500.500000,459.500000,3.827182e+05
75%,750.250000,710.250000,7.603211e+05
max,1000.000000,1000.000000,1.972127e+06


## Part 2. Splitting the dataset

In [4]:
# Take the pandas dataset and split it into our features (X) and label (y)
X = df[["Temperature °C", "Mols KCL"]]
y = df["Size nm^3"]

# Use sklearn to split the features and labels into a training/test set. (90% train, 10% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=42)

## Part 3. Perform a Linear Regression

In [12]:
# Use sklearn to train a model on the training set
model = LinearRegression()
model.fit(X_train, y_train)

# Create a sample datapoint and predict the output of that sample with the trained model
sample_datapoint = pd.DataFrame([[500, 471]], columns=["Temperature °C", "Mols KCL"])
predicted_output = model.predict(sample_datapoint)
print("Predicted output for sample datapoint:", predicted_output)

# Report on the score for that model, in your own words (markdown, not code) explain what the score means
score = model.score(X_test, y_test)
print("Model score:", score)
#In the comments below

# Extract the coefficents and intercept from the model and write an equation for your h(x) using LaTeX
coefficients = model.coef_
intercept = model.intercept_
print("Coefficients:", coefficients)
print("Intercept:", intercept)

Predicted output for sample datapoint: [510081.10341736]
Model score: 0.8552472077276095
Coefficients: [ 866.14641337 1032.69506649]
Intercept: -409391.47958340764


The predicted score of 0.855 indicates approximately 85.5% variance in the label (Size in mm^3) is predicted by the 2 features (Temperature and Mols KCL)
A high score of 0.855 (Range 0 - 1) shows that the model is able to capture the output prediction for a set of inputs.

$$
h(x) = -409391.48 + 866.15 \cdot \text{Temperature }^\circ\text{C} + 1032.70 \cdot \text{Mols KCL}
$$


## Part 4. Use Cross Validation

In [8]:
# Use the cross_val_score function to repeat your experiment across many shuffles of the data
# Report on their finding and their significance
scores = cross_val_score(model, X, y, cv=5)
print("Cross-validation scores:", scores)
print("Mean score:", np.mean(scores))

# Report on their finding and their significance

Cross-validation scores: [0.83918826 0.87051239 0.85871066 0.87202623 0.84364641]
Mean score: 0.8568167899144437


Cross-validation helps to ensure that the model is not overfitting the data by splitting the data into different sets and testing the model multiple times.
The mean score of the cross-validation represents the average performance of the model across different splits.
In this case, a higher mean score indicates that the model is performing well across various data shuffles.

## Part 5. Using Polynomial Regression

In [25]:
# Using the PolynomialFeatures library perform another regression on an augmented dataset of degree 2
poly = PolynomialFeatures(degree=2)
X_poly = poly.fit_transform(X)

# Splitting the polynomial features into training and test sets
X_poly_train, X_poly_test, y_train_poly, y_test_poly = train_test_split(X_poly, y, test_size=0.1, random_state=42)

# Training the polynomial regression model
model_poly = LinearRegression()
model_poly.fit(X_poly_train, y_train_poly)

# Report on the score for the polynomial model
score_poly = model_poly.score(X_poly_test, y_test_poly)
print("Polynomial Model score:", score_poly)

# Extract the coefficients and intercept from the polynomial model
coefficients_poly = model_poly.coef_
intercept_poly = model_poly.intercept_
print("Polynomial Model coefficients:", coefficients_poly)
print("Polynomial Model intercept:", intercept_poly)

# Display the feature names to map coefficients correctly
feature_names = poly.get_feature_names_out(X.columns)
coeff_dict = dict(zip(feature_names[1:], coefficients_poly[1:]))
print("Coefficients mapped to features:")
for feature, coeff in coeff_dict.items():
    print(f"{feature}: {coeff}")

Polynomial Model score: 1.0
Polynomial Model coefficients: [ 0.00000000e+00  1.20000000e+01 -1.27195488e-07  1.26494371e-11
  2.00000000e+00  2.85714287e-02]
Polynomial Model intercept: 2.0477105863392353e-05
Coefficients mapped to features:
Temperature °C: 11.99999997769234
Mols KCL: -1.2719548825074374e-07
Temperature °C^2: 1.2649437053369184e-11
Temperature °C Mols KCL: 2.000000000017221
Mols KCL^2: 0.028571428708643598


The polynomial regression model achieved an score of 1.0, indicating a perfect fit to the data. This means that the model explains 100% of the variance in the target variable (Size nm³) based on the input features and their polynomial combinations.

Equation
$$
h(x) = 2.047 \times 10^{-5} + 12.00 \times \text{Temperature}^\circ\text{C} + 2.00 \times \text{Temperature}^\circ\text{C} \times \text{Mols KCL} + 0.02857 \times \left(\text{Mols KCL}\right)^2
$$
